<a href="https://colab.research.google.com/github/costaraujoml-ops/Aurora-Siger-Sistema-Inteligente-da-Col-nia-/blob/main/Aurora_Siger_Sistema_Inteligente_da_Col%C3%B4nia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Aurora Siger: Sistema Inteligente da Colônia**
Sistema integrado que representa o funcionamento inteligente da colônia em Marte.

O sistema segue o fluxo **entrada → processamento → saída** e reúne quatro partes:

1. **Organização dos dados:** dicionários aninhados (chave-valor e hierarquia).
2. **Regras de decisão:** condicionais `if/elif/else` e priorização de sistemas.
3. **Previsão:** regressão linear simples (vento → energia eólica).
4. **Análise de energia:** comparação entre geração e consumo.

## 1. Organização dos dados da colônia

Os dados ficam em um **dicionário aninhado** (estrutura chave-valor com hierarquia), por exemplo: sistema energético → solar / eólico.

Esse formato permite acesso rápido por chave, como `colonia["energia"]["solar"]["geracao"]`.

Cada módulo operacional tem:
- `prioridade` (1 = mais importante);
- `essencial` (True/False), usado para nunca desligar o suporte à vida;
- `consumo` (em unidades de energia).

O `historico` guarda as leituras passadas de vento e energia eólica, usadas na previsão.

Também definimos aqui os **limites** das regras de decisão. São suposições do grupo: energia < 50 é "baixa" e consumo ≥ 60 é "alto".

In [2]:
# ---------- DADOS DA COLÔNIA (dicionário aninhado) ----------
colonia = {
    "energia": {
        "solar":    {"geracao": 15},
        "eolica":   {"geracao": 25},
        "baterias": {"carga": 40},
    },
    "ambiental": {
        "temperatura_externa": -60,
        "temperatura_interna": 21,
        "vento_atual": 10,
        "vento_previsto": 11,
        "tempestade_areia": False,
    },
    "operacional": {
        "modulos": {
            "suporte_vida":       {"prioridade": 1, "essencial": True,  "consumo": 30},
            "comunicacao":        {"prioridade": 2, "essencial": True,  "consumo": 10},
            "laboratorio":        {"prioridade": 3, "essencial": False, "consumo": 20},
            "iluminacao_externa": {"prioridade": 4, "essencial": False, "consumo": 10},
        },
    },
    "historico": {
        "vento":          [8, 10, 12],
        "energia_eolica": [20, 25, 30],
    },
}

# ---------- LIMITES USADOS NAS REGRAS ----------
LIMITE_ENERGIA_BAIXA = 50   # bateria abaixo disso é considerada baixa
LIMITE_CONSUMO_ALTO = 60    # consumo a partir disso é considerado alto

# Acesso rápido por chave
print("Geração solar:", colonia["energia"]["solar"]["geracao"])

Geração solar: 15


## 2. Funções de leitura e navegação

Aqui criamos funções para **consultar** os dados:

- `obter_geracao_total`: soma a geração solar e eólica.
- `obter_consumo_total`: soma o consumo de todos os módulos.
- `exibir_hierarquia`: percorre o dicionário e mostra a árvore de subsistemas. Usa **recursão**, ou seja, a função chama a si mesma para entrar em cada nível.

In [3]:
def obter_geracao_total(c):
    """Soma a geração solar e a eólica."""
    solar = c["energia"]["solar"]["geracao"]
    eolica = c["energia"]["eolica"]["geracao"]
    return solar + eolica


def obter_consumo_total(c):
    """Soma o consumo de todos os módulos operacionais."""
    total = 0
    for modulo in c["operacional"]["modulos"].values():
        total = total + modulo["consumo"]
    return total


def exibir_hierarquia(dados, nivel=0):
    """Mostra a hierarquia da colônia (função recursiva)."""
    for chave, valor in dados.items():
        espaco = "   " * nivel
        if isinstance(valor, dict):
            print(f"{espaco}- {chave}")
            exibir_hierarquia(valor, nivel + 1)
        else:
            print(f"{espaco}- {chave}: {valor}")


exibir_hierarquia(colonia)
print()
print("Geração total:", obter_geracao_total(colonia))
print("Consumo total:", obter_consumo_total(colonia))

- energia
   - solar
      - geracao: 15
   - eolica
      - geracao: 25
   - baterias
      - carga: 40
- ambiental
   - temperatura_externa: -60
   - temperatura_interna: 21
   - vento_atual: 10
   - vento_previsto: 11
   - tempestade_areia: False
- operacional
   - modulos
      - suporte_vida
         - prioridade: 1
         - essencial: True
         - consumo: 30
      - comunicacao
         - prioridade: 2
         - essencial: True
         - consumo: 10
      - laboratorio
         - prioridade: 3
         - essencial: False
         - consumo: 20
      - iluminacao_externa
         - prioridade: 4
         - essencial: False
         - consumo: 10
- historico
   - vento: [8, 10, 12]
   - energia_eolica: [20, 25, 30]

Geração total: 40
Consumo total: 70


## 3. Regras de decisão (lógica do sistema)

A função `decidir_acao` transforma dados em uma **decisão clara**, combinando três critérios com AND:

- energia baixa (bateria < 50);
- consumo alto (≥ 60);
- previsão desfavorável.

As regras são avaliadas **da mais grave para a menos grave**:

1. Energia baixa **E** consumo alto **E** previsão desfavorável → **EMERGÊNCIA**.
2. Energia baixa **E** consumo alto → **ALERTA: reduzir consumo**.
3. Apenas energia baixa → **ATENÇÃO: ativar modo economia**.
4. Caso contrário → **OK**.

Isso segue o fluxo da Figura 1 do enunciado: se a energia não é suficiente, ativa o modo de emergência.

In [4]:
def decidir_acao(energia, consumo, previsao_desfavoravel=False):
    """Decide a ação da colônia a partir de energia, consumo e previsão."""
    energia_baixa = energia < LIMITE_ENERGIA_BAIXA
    consumo_alto = consumo >= LIMITE_CONSUMO_ALTO

    if energia_baixa and consumo_alto and previsao_desfavoravel:
        return "EMERGÊNCIA: ativar modo de emergência"
    elif energia_baixa and consumo_alto:
        return "ALERTA: reduzir consumo"
    elif energia_baixa:
        return "ATENÇÃO: ativar modo economia"
    else:
        return "OK: manter sistemas normais"


# Exemplo do enunciado: energia = 40, consumo = 70
print(decidir_acao(40, 70))

ALERTA: reduzir consumo


## 4. Priorização de sistemas

Quando é preciso economizar energia, o sistema **mantém o suporte à vida ligado** e desliga primeiro o que é menos importante.

- `sistemas_nao_essenciais`: lista os módulos não essenciais, do **menos importante para o mais importante** (maior número de prioridade primeiro).
- `aplicar_modo_economia`: desliga esses módulos e calcula o consumo que sobra, só com os módulos essenciais.

Módulos com `essencial = True` (como o suporte à vida) **nunca** entram na lista de desligamento.

In [5]:
def chave_prioridade(item):
    """Auxiliar para ordenar: usa o número de prioridade do módulo."""
    return item[1]["prioridade"]


def sistemas_nao_essenciais(modulos):
    """Retorna os módulos não essenciais, do menos para o mais importante."""
    lista = []
    for nome, dados in modulos.items():
        if not dados["essencial"]:
            lista.append((nome, dados))
    lista = sorted(lista, key=chave_prioridade, reverse=True)
    return [nome for nome, dados in lista]


def aplicar_modo_economia(modulos):
    """Desliga os não essenciais e retorna (desligados, consumo_restante)."""
    desligados = sistemas_nao_essenciais(modulos)
    consumo_restante = 0
    for dados in modulos.values():
        if dados["essencial"]:
            consumo_restante = consumo_restante + dados["consumo"]
    return desligados, consumo_restante


desligados, restante = aplicar_modo_economia(colonia["operacional"]["modulos"])
print("Desligar (nesta ordem):", desligados)
print("Consumo restante:", restante)

Desligar (nesta ordem): ['iluminacao_externa', 'laboratorio']
Consumo restante: 40


## 5. Análise do uso de energia

A função `analisar_energia` compara **geração** e **consumo** e devolve uma ação simples:

- consumo > geração → **ALERTA** (risco);
- geração > consumo → **SUGESTÃO** de armazenar o excedente (possível desperdício);
- iguais → equilíbrio.

Os dois exemplos abaixo são os do enunciado.

In [6]:
def analisar_energia(geracao, consumo):
    """Compara geração e consumo e retorna uma mensagem."""
    if consumo > geracao:
        return "ALERTA: consumo maior que geração"
    elif geracao > consumo:
        return "SUGESTÃO: armazenar energia excedente"
    else:
        return "EQUILÍBRIO: geração igual ao consumo"


print(analisar_energia(40, 70))  # esperado: ALERTA
print(analisar_energia(80, 30))  # esperado: SUGESTÃO

ALERTA: consumo maior que geração
SUGESTÃO: armazenar energia excedente


## 6. Previsão com regressão linear simples

Modelo: **Y = b0 + b1 · X**, em que X é o vento e Y é a energia eólica gerada.

Os coeficientes são calculados pelo **método dos mínimos quadrados**:

- b1 = Σ(x − x̄)(y − ȳ) / Σ(x − x̄)²
- b0 = ȳ − b1 · x̄

Funções:
- `regressao_linear`: calcula b0 e b1 a partir das listas de histórico.
- `prever`: usa a reta para estimar Y para um novo X.
- `calcular_r2`: mede a qualidade do ajuste (quanto mais perto de 1, melhor).

Exemplo do enunciado: vento = [8, 10, 12] e energia = [20, 25, 30]. Para vento = 11, a previsão é 27,5, o que corresponde ao "≈ 27" do enunciado.

In [7]:
def regressao_linear(x, y):
    """Retorna (b0, b1) da reta ajustada por mínimos quadrados."""
    n = len(x)
    x_med = sum(x) / n
    y_med = sum(y) / n

    numerador = 0
    denominador = 0
    for xi, yi in zip(x, y):
        numerador = numerador + (xi - x_med) * (yi - y_med)
        denominador = denominador + (xi - x_med) ** 2

    if denominador == 0:
        return y_med, 0.0  # sem variação em x: reta horizontal

    b1 = numerador / denominador
    b0 = y_med - b1 * x_med
    return b0, b1


def prever(b0, b1, x_novo):
    """Estima Y para um novo valor de X."""
    return b0 + b1 * x_novo


def calcular_r2(x, y, b0, b1):
    """Coeficiente de determinação (R²)."""
    y_med = sum(y) / len(y)
    soma_erros = 0
    soma_total = 0
    for xi, yi in zip(x, y):
        soma_erros = soma_erros + (yi - prever(b0, b1, xi)) ** 2
        soma_total = soma_total + (yi - y_med) ** 2
    if soma_total == 0:
        return 1.0
    return 1 - soma_erros / soma_total


vento = colonia["historico"]["vento"]
energia = colonia["historico"]["energia_eolica"]

b0, b1 = regressao_linear(vento, energia)
print(f"Modelo: energia = {b0:.2f} + {b1:.2f} * vento")
print(f"R² = {calcular_r2(vento, energia, b0, b1):.2f}")
print(f"Previsão para vento = 11: {prever(b0, b1, 11):.1f}")

Modelo: energia = 0.00 + 2.50 * vento
R² = 1.00
Previsão para vento = 11: 27.5


## 7. Sistema integrado

A função `executar_sistema` junta todas as partes seguindo o fluxo **entrada → processamento → saída**:

1. **Entrada:** lê bateria, geração, consumo e vento previsto no dicionário da colônia.
2. **Processamento:**
   - ajusta a regressão e prevê a geração eólica;
   - calcula a geração prevista (solar + eólica prevista);
   - define se a previsão é desfavorável (tempestade de areia **ou** geração prevista menor que o consumo);
   - decide a ação e analisa o uso de energia.
3. **Saída:** mostra o relatório e, se necessário, os módulos a desligar.

A previsão eólica usa `max(0, ...)` para nunca ser negativa.

In [8]:
def executar_sistema(c):
    """Executa o ciclo completo: entrada -> processamento -> saída."""
    # ----- ENTRADA -----
    bateria = c["energia"]["baterias"]["carga"]
    geracao = obter_geracao_total(c)
    consumo = obter_consumo_total(c)
    ambiente = c["ambiental"]

    # ----- PROCESSAMENTO -----
    b0, b1 = regressao_linear(c["historico"]["vento"],
                              c["historico"]["energia_eolica"])
    eolica_prevista = max(0, prever(b0, b1, ambiente["vento_previsto"]))
    geracao_prevista = c["energia"]["solar"]["geracao"] + eolica_prevista

    previsao_desfavoravel = (ambiente["tempestade_areia"]
                             or geracao_prevista < consumo)

    acao = decidir_acao(bateria, consumo, previsao_desfavoravel)
    analise = analisar_energia(geracao, consumo)

    # ----- SAÍDA -----
    print("=== RELATÓRIO DA COLÔNIA AURORA SIGER ===")
    print(f"Bateria: {bateria} | Geração: {geracao} | Consumo: {consumo}")
    print(f"Vento previsto: {ambiente['vento_previsto']} "
          f"-> energia eólica prevista: {eolica_prevista:.1f}")
    print(f"Geração prevista total: {geracao_prevista:.1f}")
    print(f"Previsão desfavorável: {previsao_desfavoravel}")
    print("Decisão:", acao)
    print("Análise de energia:", analise)

    if not acao.startswith("OK"):
        desligar, restante = aplicar_modo_economia(c["operacional"]["modulos"])
        print("Suporte à vida: MANTIDO LIGADO")
        print("Desligar sistemas não essenciais:", desligar)
        print("Consumo após economia:", restante)

    return acao


executar_sistema(colonia)

=== RELATÓRIO DA COLÔNIA AURORA SIGER ===
Bateria: 40 | Geração: 40 | Consumo: 70
Vento previsto: 11 -> energia eólica prevista: 27.5
Geração prevista total: 42.5
Previsão desfavorável: True
Decisão: EMERGÊNCIA: ativar modo de emergência
Análise de energia: ALERTA: consumo maior que geração
Suporte à vida: MANTIDO LIGADO
Desligar sistemas não essenciais: ['iluminacao_externa', 'laboratorio']
Consumo após economia: 40


'EMERGÊNCIA: ativar modo de emergência'

## 8. Testes com os exemplos do enunciado

Conferimos se o sistema produz as saídas esperadas:

| Entrada | Saída esperada |
|---|---|
| energia = 40, consumo = 70 | ALERTA: reduzir consumo |
| geração = 40, consumo = 70 | ALERTA: consumo maior que geração |
| geração = 80, consumo = 30 | SUGESTÃO: armazenar energia excedente |
| vento = 11 | energia ≈ 27 (o modelo dá 27,5) |

Se algum teste falhar, aparece uma mensagem de erro.

In [9]:
# Teste 1: regra de decisão
assert decidir_acao(40, 70) == "ALERTA: reduzir consumo"

# Teste 2 e 3: análise de energia
assert analisar_energia(40, 70) == "ALERTA: consumo maior que geração"
assert analisar_energia(80, 30) == "SUGESTÃO: armazenar energia excedente"

# Teste 4: previsão (vento = 11 -> ≈ 27)
b0, b1 = regressao_linear([8, 10, 12], [20, 25, 30])
previsao = prever(b0, b1, 11)
assert 27 <= previsao <= 28

# Teste 5: o suporte à vida nunca é desligado
desligados, _ = aplicar_modo_economia(colonia["operacional"]["modulos"])
assert "suporte_vida" not in desligados

print("Todos os testes passaram!")

Todos os testes passaram!
